# OpenST preprocessing

This public notebook was migrated from the audited read-only research source. Configure paths in `config.yaml` before execution. Time is metadata and is never a model input.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'OpenST'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import numpy as np
import anndata as ad
import scanpy as sc
import scvi
import sys


In [ ]:
base_dir = str(DATA_ROOT / 'raw')
samples = {
    "S2": "Reconstructed_S2.h5ad",
    "S3": "Reconstructed_S3.h5ad",
    "S4": "Reconstructed_S4.h5ad",
    "S5": "Reconstructed_S5.h5ad",
    "S6": "Reconstructed_S6.h5ad",
    "S7": "Reconstructed_S7.h5ad",
}
adatas = {}

for sample_id, filename in samples.items():
    path = os.path.join(base_dir, filename)
    sample_adata = sc.read_h5ad(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata
adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()
print(adata.obs["sample"].value_counts())
adata

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000, batch_key="sample")

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
sc.tl.pca(adata)

In [ ]:
sc.pl.pca(
    adata,
    color=["sample", "sample", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color="sample", size=2)

In [ ]:
sc.pl.umap(adata, color="annotation", size=2)

In [ ]:
scvi.model.SCANVI.setup_anndata(
    adata,
    layer="raw",
    labels_key="annotation",
    unlabeled_category="Unknown",   
    batch_key="sample"
)

In [ ]:
model = scvi.model.SCANVI(adata)

In [ ]:
model

In [ ]:
model.train()

In [ ]:
model_dir = SCANVI_DIR


In [ ]:
model


In [ ]:
SCVI_LATENT_KEY = "X_scanVI"

latent = model.get_latent_representation()
adata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
# run PCA then generate UMAP plots
sc.tl.pca(adata)
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata,color=["annotation"],frameon=False,size = 5,)

In [ ]:
adata.obs["cx_aligned"] = adata.obsm['spatial_3d_aligned'][:, 0].astype(np.float32)
adata.obs["cy_aligned"] = adata.obsm['spatial_3d_aligned'][:, 1].astype(np.float32)

In [ ]:
model.save(model_dir, overwrite=True, save_anndata=True)
print(f"Saved model-ready AnnData: {SCANVI_ADATA}")


In [ ]:
adata